# Member 1 - Decision Tree Regressor
Train and compare against ElasticNet baseline.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

root = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
data_path = root / 'data' / 'dataset_finalized_region9_weekly_8features.csv'
models_dir = root / 'models'
models_dir.mkdir(exist_ok=True)
feature_cols = [
    'feat_mean_tmax_c_week','feat_max_tmax_c_week','feat_temp_range_c_week','feat_heat_intensity',
    'feat_poverty_rate','feat_unemployment_rate','feat_median_hh_income','feat_total_population'
]
target_col = 'target_hri_rate'


In [ ]:
df = pd.read_csv(data_path).dropna(subset=feature_cols+[target_col])
X = df[feature_cols]
y = df[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeRegressor(max_depth=5, min_samples_leaf=4, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)
metrics = {
    'model': 'DecisionTreeRegressor',
    'params': {'max_depth': 5, 'min_samples_leaf': 4, 'random_state': 42},
    'mae': float(mean_absolute_error(y_test, preds)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, preds))),
    'r2': float(r2_score(y_test, preds)),
}
metrics


In [ ]:
joblib.dump(model, models_dir / 'member1_decision_tree.pkl')
(models_dir / 'member1_decision_tree_metrics.json').write_text(json.dumps(metrics, indent=2))

elastic_path = models_dir / 'member1_metrics.json'
if elastic_path.exists():
    elastic = json.loads(elastic_path.read_text())
    comparison = pd.DataFrame([
        {'model':'ElasticNet','MAE':elastic.get('mae'),'RMSE':elastic.get('rmse'),'R2':elastic.get('r2')},
        {'model':'DecisionTree','MAE':metrics['mae'],'RMSE':metrics['rmse'],'R2':metrics['r2']}
    ])
    display(comparison)
else:
    print('ElasticNet metrics file not found')
